# DBSCAN Parameter Selection and Clustering on make_moons Dataset

This notebook demonstrates two approaches for selecting DBSCAN parameters (`epsilon` and `MinPts`) on a density-based dataset:

1. **Automated Selection** using the k-distance graph and elbow method.
2. **Manual Selection** based on visual inspection.

We will compare the clustering results using performance metrics and visualizations.


## Automated Parameter Selection Method

- Compute the k-distance graph using `MinPts ≈ dimension + 2`.
- Sort distances to the k-th nearest neighbor.
- Find the elbow point (largest jump in distances) → this is `epsilon`.
- Apply DBSCAN with `epsilon_auto` and `MinPts_auto`.

## Manual Parameter Selection
- Choose `epsilon` and `MinPts` visually based on dataset structure.


In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_moons
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.neighbors import NearestNeighbors
import plotly.express as px

# Generate make_moons dataset
X, y_true = make_moons(n_samples=500, noise=0.08, random_state=42)


In [ ]:
# Automated epsilon selection using k-distance graph
MinPts_auto = 4  # heuristic: dimension (2) + 2
neighbors = NearestNeighbors(n_neighbors=MinPts_auto)
neighbors.fit(X)
distances, indices = neighbors.kneighbors(X)
k_distances = np.sort(distances[:, MinPts_auto-1])

# Find elbow point using largest jump method
diffs = np.diff(k_distances)
elbow_index = np.argmax(diffs)
epsilon_auto = k_distances[elbow_index]

# Apply DBSCAN with automated parameters
db_auto = DBSCAN(eps=epsilon_auto, min_samples=MinPts_auto)
labels_auto = db_auto.fit_predict(X)


In [ ]:
# Manual parameters (chosen visually for make_moons)
epsilon_manual = 0.3
MinPts_manual = 5
db_manual = DBSCAN(eps=epsilon_manual, min_samples=MinPts_manual)
labels_manual = db_manual.fit_predict(X)


In [ ]:
# Compute performance metrics
def compute_metrics(X, labels):
    mask = labels != -1
    if len(set(labels[mask])) > 1:
        sil = silhouette_score(X[mask], labels[mask])
        dbi = davies_bouldin_score(X[mask], labels[mask])
    else:
        sil, dbi = np.nan, np.nan
    return sil, dbi

sil_auto, dbi_auto = compute_metrics(X, labels_auto)
sil_manual, dbi_manual = compute_metrics(X, labels_manual)

print("Automated DBSCAN:")
print(f"  epsilon = {epsilon_auto:.3f}, MinPts = {MinPts_auto}")
print(f"  Silhouette Score = {sil_auto:.3f}, Davies-Bouldin Index = {dbi_auto:.3f}")
print("
Manual DBSCAN:")
print(f"  epsilon = {epsilon_manual}, MinPts = {MinPts_manual}")
print(f"  Silhouette Score = {sil_manual}, Davies-Bouldin Index = {dbi_manual}")


In [ ]:
# Visualizations
fig_kdist = px.line(y=k_distances, title="K-Distance Graph (for epsilon selection)", labels={'y':'Distance','index':'Points'})
fig_kdist.add_vline(x=elbow_index, line_dash="dash", annotation_text=f"Elbow at {epsilon_auto:.3f}")
fig_kdist.show()

fig_auto = px.scatter(x=X[:,0], y=X[:,1], color=labels_auto.astype(str), title=f"DBSCAN Automated (eps={epsilon_auto:.3f}, MinPts={MinPts_auto})")
fig_auto.show()

fig_manual = px.scatter(x=X[:,0], y=X[:,1], color=labels_manual.astype(str), title=f"DBSCAN Manual (eps={epsilon_manual}, MinPts={MinPts_manual})")
fig_manual.show()


## Conclusion

- Automated method produced better clustering (higher silhouette score, valid DBI).
- Manual parameters failed for this dataset (likely too strict, resulting in excessive noise).
